# GFI v2.0 — Analisis & Eksplorasi Hasil

Notebook ini digunakan **setelah** menjalankan `GFI2_Colab.ipynb`.  
Memuat hasil dari folder `output/` dan menyediakan analisis lanjutan:
- Perbandingan spasial GFI v1.0 vs v2.0
- Analisis distribusi kedalaman banjir
- Ekspor statistik ke CSV
- Visualisasi kustom per area of interest

In [ ]:
# ── SEL 1: Import ─────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import os, sys

# Pastikan sudah clone repo dan ada di path
# !git clone https://github.com/YOUR_USERNAME/gfi2-python.git
sys.path.insert(0, 'gfi2-python')

from gfi2 import load_tif, save_tif, compute_validation_metrics

print('Import berhasil.')

In [ ]:
# ── SEL 2: Muat hasil dari folder output ──────────────────────────────────
OUT_DIR = 'output'   # sesuaikan jika berbeda

GFIv1, profile = load_tif(f'{OUT_DIR}/GFI_v1.tif')
GFIv2, _       = load_tif(f'{OUT_DIR}/GFI_v2.tif')
WDv1,  _       = load_tif(f'{OUT_DIR}/WD_v1.tif')
WDv2,  _       = load_tif(f'{OUT_DIR}/WD_v2.tif')

print(f'Shape raster : {GFIv1.shape}')
print(f'GFIv1 range  : [{np.nanmin(GFIv1):.3f}, {np.nanmax(GFIv1):.3f}]')
print(f'GFIv2 range  : [{np.nanmin(GFIv2):.3f}, {np.nanmax(GFIv2):.3f}]')
print(f'WDv2  range  : [{np.nanmin(WDv2):.3f}, {np.nanmax(WDv2):.3f}] m')

In [ ]:
# ── SEL 3: Peta perbandingan GFI v1.0 vs v2.0 ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmin = min(np.nanmin(GFIv1), np.nanmin(GFIv2))
vmax = max(np.nanmax(GFIv1), np.nanmax(GFIv2))

im1 = axes[0].imshow(GFIv1, cmap='RdYlBu', vmin=vmin, vmax=vmax)
plt.colorbar(im1, ax=axes[0], label='GFI')
axes[0].set_title('GFI v1.0')
axes[0].axis('off')

im2 = axes[1].imshow(GFIv2, cmap='RdYlBu', vmin=vmin, vmax=vmax)
plt.colorbar(im2, ax=axes[1], label='GFI')
axes[1].set_title('GFI v2.0 (confluence-aware)')
axes[1].axis('off')

# Peta selisih
diff = GFIv2 - GFIv1
lim  = np.nanpercentile(np.abs(diff), 98)
im3  = axes[2].imshow(diff, cmap='bwr', vmin=-lim, vmax=lim)
plt.colorbar(im3, ax=axes[2], label='ΔGFI (v2.0 − v1.0)')
axes[2].set_title('Selisih GFI')
axes[2].axis('off')

plt.suptitle('Perbandingan GFI v1.0 vs v2.0', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/comparison_GFI.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SEL 4: Peta perbandingan Water Depth ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

wmax = max(np.nanmax(WDv1), np.nanmax(WDv2))

im1 = axes[0].imshow(WDv1, cmap='Blues', vmin=0, vmax=wmax)
plt.colorbar(im1, ax=axes[0], label='WD [m]')
axes[0].set_title('Water Depth — GFI v1.0')
axes[0].axis('off')

im2 = axes[1].imshow(WDv2, cmap='Blues', vmin=0, vmax=wmax)
plt.colorbar(im2, ax=axes[1], label='WD [m]')
axes[1].set_title('Water Depth — GFI v2.0')
axes[1].axis('off')

diff_wd = WDv2 - WDv1
lim_wd  = np.nanpercentile(np.abs(diff_wd), 98)
im3     = axes[2].imshow(diff_wd, cmap='bwr', vmin=-lim_wd, vmax=lim_wd)
plt.colorbar(im3, ax=axes[2], label='ΔWD [m]')
axes[2].set_title('Selisih Water Depth')
axes[2].axis('off')

plt.suptitle('Perbandingan Water Depth v1.0 vs v2.0', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/comparison_WD.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SEL 5: Distribusi nilai GFI dan WD ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram GFI
v1_flat = GFIv1[~np.isnan(GFIv1)].ravel()
v2_flat = GFIv2[~np.isnan(GFIv2)].ravel()
bins_gfi = np.linspace(min(v1_flat.min(), v2_flat.min()),
                        max(v1_flat.max(), v2_flat.max()), 80)

axes[0].hist(v1_flat, bins=bins_gfi, alpha=0.5, color='steelblue',
             label='GFI v1.0', density=True)
axes[0].hist(v2_flat, bins=bins_gfi, alpha=0.5, color='tomato',
             label='GFI v2.0', density=True)
axes[0].axvline(np.nanmedian(GFIv1), color='steelblue', ls='--', lw=1.5,
                label=f'Median v1.0 = {np.nanmedian(GFIv1):.2f}')
axes[0].axvline(np.nanmedian(GFIv2), color='tomato', ls='--', lw=1.5,
                label=f'Median v2.0 = {np.nanmedian(GFIv2):.2f}')
axes[0].set_xlabel('GFI value')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribusi GFI')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Histogram WD (hanya piksel > 0)
wd1_flat = WDv1[WDv1 > 0].ravel()
wd2_flat = WDv2[WDv2 > 0].ravel()

if len(wd1_flat) > 0 and len(wd2_flat) > 0:
    bins_wd = np.linspace(0, max(wd1_flat.max(), wd2_flat.max()), 60)
    axes[1].hist(wd1_flat, bins=bins_wd, alpha=0.5, color='steelblue',
                 label='WD v1.0', density=True)
    axes[1].hist(wd2_flat, bins=bins_wd, alpha=0.5, color='tomato',
                 label='WD v2.0', density=True)
    axes[1].axvline(np.nanmedian(wd1_flat), color='steelblue', ls='--', lw=1.5,
                    label=f'Median v1.0 = {np.nanmedian(wd1_flat):.2f} m')
    axes[1].axvline(np.nanmedian(wd2_flat), color='tomato', ls='--', lw=1.5,
                    label=f'Median v2.0 = {np.nanmedian(wd2_flat):.2f} m')
    axes[1].set_xlabel('Water Depth [m]')
    axes[1].set_ylabel('Density')
    axes[1].set_title('Distribusi Water Depth (area banjir)')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/distribution_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SEL 6: Statistik deskriptif → CSV ────────────────────────────────────
def describe_raster(arr, name):
    flat = arr[~np.isnan(arr)].ravel()
    flooded = arr[arr > 0].ravel() if name.startswith('WD') else flat
    return {
        'Variabel'  : name,
        'N piksel'  : len(flat),
        'Min'       : float(np.nanmin(flat)),
        'Max'       : float(np.nanmax(flat)),
        'Mean'      : float(np.nanmean(flat)),
        'Median'    : float(np.nanmedian(flat)),
        'Std'       : float(np.nanstd(flat)),
        'P25'       : float(np.nanpercentile(flat, 25)),
        'P75'       : float(np.nanpercentile(flat, 75)),
        'Flooded N' : int((arr > 0).sum()) if name.startswith('WD') else '-',
    }

stats = pd.DataFrame([
    describe_raster(GFIv1, 'GFI v1.0'),
    describe_raster(GFIv2, 'GFI v2.0'),
    describe_raster(WDv1,  'WD v1.0 [m]'),
    describe_raster(WDv2,  'WD v2.0 [m]'),
])

csv_path = f'{OUT_DIR}/descriptive_statistics.csv'
stats.to_csv(csv_path, index=False, float_format='%.4f')
print(f'Statistik tersimpan: {csv_path}')
stats

In [ ]:
# ── SEL 7: Download semua output analisis ─────────────────────────────────
from google.colab import files

extra_outputs = [
    'comparison_GFI.png',
    'comparison_WD.png',
    'distribution_analysis.png',
    'descriptive_statistics.csv',
]

for fname in extra_outputs:
    fpath = f'{OUT_DIR}/{fname}'
    if os.path.exists(fpath):
        print(f'Mendownload: {fname}')
        files.download(fpath)